In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 140)
pd.set_option('display.width', 180)

### 1. Đọc dữ liệu và kiểm tra tổng quan

In [2]:
DATA_PATH = "../../data/dataset2/dataset2_full.csv"
df_raw = pd.read_csv(DATA_PATH, low_memory=False)

print('Dataset 2 full shape:', df_raw.shape)
print('Columns:', len(df_raw.columns))
display(pd.DataFrame({'column': df_raw.columns, 'dtype': [df_raw[c].dtype for c in df_raw.columns]}))
display(df_raw.head())

Dataset 2 full shape: (149116, 47)
Columns: 47


,column,dtype
0,transaction_id,int64
1,transaction_date,object
2,transaction_time,object
3,transaction_qty,int64
4,store_id,int64
5,store_location,object
6,product_id,int64
7,unit_price,float64
8,product_category,object
9,product_type,object


,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail,transaction_datetime,date,hour,location,total_amount,city,borough,temperature_c,apparent_temperature_c,relative_humidity,precipitation_mm,rain_mm,weather_code,weather_code_description,weather_condition,wind_speed_10m_kmh,cloud_cover_pct,is_rainy,is_snowy,is_precipitation,weather_latitude,weather_longitude,weather_source,weather_spatial_level,holiday_name,holiday_type,is_official_holiday,is_commercial_event,notes,is_holiday,is_pre_holiday,is_post_holiday,days_to_nearest_holiday,nearest_holiday_name,holiday_source,holiday_calendar_scope
0,1,2023-01-01,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg,2023-01-01 07:06:11,2023-01-01,7,Lower Manhattan,6.0,New York,Manhattan,7.9,5.2,94,0.0,0.0,2,Partly cloudy,Cloudy,13.2,58,False,False,False,40.7075,-74.0113,Open-Meteo Historical Weather API,area,New Year's Day,official,True,False,Actual New Year's Day,True,True,False,0,New Year's Day,data/holiday/nyc_2023_holiday_calendar.csv,NYC/US 2023 official holidays and selected com...
1,2,2023-01-01,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,2023-01-01 07:08:56,2023-01-01,7,Lower Manhattan,6.2,New York,Manhattan,7.9,5.2,94,0.0,0.0,2,Partly cloudy,Cloudy,13.2,58,False,False,False,40.7075,-74.0113,Open-Meteo Historical Weather API,area,New Year's Day,official,True,False,Actual New Year's Day,True,True,False,0,New Year's Day,data/holiday/nyc_2023_holiday_calendar.csv,NYC/US 2023 official holidays and selected com...
2,3,2023-01-01,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg,2023-01-01 07:14:04,2023-01-01,7,Lower Manhattan,9.0,New York,Manhattan,7.9,5.2,94,0.0,0.0,2,Partly cloudy,Cloudy,13.2,58,False,False,False,40.7075,-74.0113,Open-Meteo Historical Weather API,area,New Year's Day,official,True,False,Actual New Year's Day,True,True,False,0,New Year's Day,data/holiday/nyc_2023_holiday_calendar.csv,NYC/US 2023 official holidays and selected com...
3,4,2023-01-01,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm,2023-01-01 07:20:24,2023-01-01,7,Lower Manhattan,2.0,New York,Manhattan,7.9,5.2,94,0.0,0.0,2,Partly cloudy,Cloudy,13.2,58,False,False,False,40.7075,-74.0113,Open-Meteo Historical Weather API,area,New Year's Day,official,True,False,Actual New Year's Day,True,True,False,0,New Year's Day,data/holiday/nyc_2023_holiday_calendar.csv,NYC/US 2023 official holidays and selected com...
4,5,2023-01-01,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,2023-01-01 07:22:41,2023-01-01,7,Lower Manhattan,6.2,New York,Manhattan,7.9,5.2,94,0.0,0.0,2,Partly cloudy,Cloudy,13.2,58,False,False,False,40.7075,-74.0113,Open-Meteo Historical Weather API,area,New Year's Day,official,True,False,Actual New Year's Day,True,True,False,0,New Year's Day,data/holiday/nyc_2023_holiday_calendar.csv,NYC/US 2023 official holidays and selected com...


In [5]:
df_raw["product_category"].nunique()

9

In [6]:
df_raw["product_category"].value_counts()

product_category
Coffee                58416
Tea                   45449
Bakery                22796
Drinking Chocolate    11468
Flavours               6790
Coffee beans           1753
Loose Tea              1210
Branded                 747
Packaged Chocolate      487
Name: count, dtype: int64

In [ ]:
missing_summary = pd.DataFrame({
    'missing_count': df_raw.isna().sum(),
    'missing_pct': df_raw.isna().mean() * 100,
}).sort_values('missing_count', ascending=False)

print('Duplicate rows:', df_raw.duplicated().sum())
print('Duplicate transaction_id:', df_raw['transaction_id'].duplicated().sum() if 'transaction_id' in df_raw.columns else 'N/A')
display(missing_summary.head(25))

## 2. Helper functions

In [ ]:
def to_bool(series, default=False):
    mapping = {
        'true': True,
        'false': False,
        '1': True,
        '0': False,
        'yes': True,
        'no': False,
        'y': True,
        'n': False,
    }
    if series.dtype == bool:
        return series.fillna(default).astype(bool)
    return series.astype(str).str.strip().str.lower().map(mapping).fillna(default).astype(bool)

def first_mode(series):
    non_missing = series.dropna()
    if non_missing.empty:
        return pd.NA
    mode_values = non_missing.mode()
    return mode_values.iloc[0] if not mode_values.empty else non_missing.iloc[0]

def iqr_bounds(series, multiplier=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - multiplier * iqr, q3 + multiplier * iqr

## 3. Chuẩn hóa schema transaction-level

Dataset 2 có địa điểm ở cấp area/store location. Để dùng chung schema với Dataset 1, ta tạo:

```text
city = New York
area = Astoria / Hell's Kitchen / Lower Manhattan
location_level = area
```

In [ ]:
df = df_raw.copy()

df['source_dataset'] = 'dataset2'
df['transaction_datetime'] = pd.to_datetime(df['transaction_datetime'], errors='coerce')
df['date'] = df['transaction_datetime'].dt.normalize()
df['month'] = df['transaction_datetime'].dt.month
df['weekday'] = df['transaction_datetime'].dt.day_name()
df['weekday_num'] = df['transaction_datetime'].dt.dayofweek
df['hour'] = df['transaction_datetime'].dt.hour
df['is_weekend'] = df['weekday_num'].isin([5, 6])

df['city'] = df['city'].fillna('New York')
df['area'] = df['store_location'].fillna(df['location'])
df['location_level'] = 'area'
df['location'] = df['area']

df['product_category'] = df['product_category'].astype(str).str.strip()
df['product_detail_clean'] = df['product_detail'].astype(str).str.strip() if 'product_detail' in df.columns else pd.NA

df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')
df['quantity'] = pd.to_numeric(df['transaction_qty'], errors='coerce')
df['revenue'] = pd.to_numeric(df['total_amount'], errors='coerce')
df['transaction_count'] = 1

numeric_external_cols = [
    'temperature_c',
    'apparent_temperature_c',
    'relative_humidity',
    'precipitation_mm',
    'rain_mm',
    'weather_code',
    'wind_speed_10m_kmh',
    'cloud_cover_pct',
    'days_to_nearest_holiday',
]
for col in numeric_external_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

bool_cols = [
    'is_rainy',
    'is_snowy',
    'is_precipitation',
    'is_holiday',
    'is_official_holiday',
    'is_commercial_event',
    'is_pre_holiday',
    'is_post_holiday',
]
for col in bool_cols:
    if col in df.columns:
        df[col] = to_bool(df[col])

common_transaction_cols = [
    'source_dataset',
    'transaction_id',
    'transaction_datetime',
    'date',
    'month',
    'weekday',
    'weekday_num',
    'hour',
    'is_weekend',
    'city',
    'area',
    'location_level',
    'location',
    'store_id',
    'store_location',
    'product_category',
    'product_detail_clean',
    'unit_price',
    'quantity',
    'transaction_count',
    'revenue',
    'temperature_c',
    'apparent_temperature_c',
    'relative_humidity',
    'precipitation_mm',
    'rain_mm',
    'weather_condition',
    'weather_code',
    'wind_speed_10m_kmh',
    'cloud_cover_pct',
    'is_rainy',
    'is_snowy',
    'is_precipitation',
    'weather_spatial_level',
    'is_holiday',
    'holiday_name',
    'holiday_type',
    'is_official_holiday',
    'is_commercial_event',
    'is_pre_holiday',
    'is_post_holiday',
    'days_to_nearest_holiday',
    'nearest_holiday_name',
]

df2_clean = df[common_transaction_cols].copy()

print('Dataset 2 clean transaction-level shape:', df2_clean.shape)
display(df2_clean.head())

## 4. Kiểm tra chất lượng sau chuẩn hóa

In [ ]:
key_cols = ['transaction_datetime', 'date', 'hour', 'city', 'area', 'product_category', 'revenue']
quality_checks = {
    'rows': len(df2_clean),
    'duplicate_transaction_id': int(df2_clean['transaction_id'].duplicated().sum()),
    'invalid_transaction_datetime': int(df2_clean['transaction_datetime'].isna().sum()),
    'missing_key_values': int(df2_clean[key_cols].isna().sum().sum()),
    'non_positive_revenue': int((df2_clean['revenue'] <= 0).sum()),
    'non_positive_quantity': int((df2_clean['quantity'] <= 0).sum()),
    'non_positive_unit_price': int((df2_clean['unit_price'] <= 0).sum()),
    'missing_weather_condition': int(df2_clean['weather_condition'].isna().sum()),
    'missing_temperature_c': int(df2_clean['temperature_c'].isna().sum()),
    'missing_holiday_name': int(df2_clean['holiday_name'].isna().sum()),
}
display(pd.Series(quality_checks, name='value').to_frame())

display(
    pd.DataFrame({
        'missing_count': df2_clean.isna().sum(),
        'missing_pct': df2_clean.isna().mean() * 100,
    }).sort_values('missing_count', ascending=False).head(20)
)

In [ ]:
revenue_lower, revenue_upper = iqr_bounds(df2_clean['revenue'])
unit_price_lower, unit_price_upper = iqr_bounds(df2_clean['unit_price'])

outlier_summary = pd.DataFrame(
    [
        {
            'field': 'revenue',
            'lower_bound': revenue_lower,
            'upper_bound': revenue_upper,
            'outlier_count': int(((df2_clean['revenue'] < revenue_lower) | (df2_clean['revenue'] > revenue_upper)).sum()),
        },
        {
            'field': 'unit_price',
            'lower_bound': unit_price_lower,
            'upper_bound': unit_price_upper,
            'outlier_count': int(((df2_clean['unit_price'] < unit_price_lower) | (df2_clean['unit_price'] > unit_price_upper)).sum()),
        },
    ]
)

display(outlier_summary)
display(df2_clean[['revenue', 'quantity', 'unit_price', 'temperature_c', 'precipitation_mm']].describe())

## 5. Aggregate về cấp mô hình

Không nên train ở transaction-level nếu mục tiêu là dự đoán doanh thu theo external factors. Cấp hợp lý hơn:

```text
date + hour + city + area + product_category
```

Dataset 2 có `area = Astoria / Hell's Kitchen / Lower Manhattan`.

In [ ]:
group_cols = [
    'source_dataset',
    'date',
    'month',
    'weekday',
    'weekday_num',
    'hour',
    'is_weekend',
    'city',
    'area',
    'location_level',
    'product_category',
]

df2_hourly = (
    df2_clean.groupby(group_cols, dropna=False)
    .agg(
        revenue=('revenue', 'sum'),
        quantity=('quantity', 'sum'),
        transaction_count=('transaction_count', 'sum'),
        avg_transaction_value=('revenue', 'mean'),
        avg_unit_price=('unit_price', 'mean'),
        temperature_c=('temperature_c', 'mean'),
        apparent_temperature_c=('apparent_temperature_c', 'mean'),
        relative_humidity=('relative_humidity', 'mean'),
        precipitation_mm=('precipitation_mm', 'mean'),
        wind_speed_10m_kmh=('wind_speed_10m_kmh', 'mean'),
        cloud_cover_pct=('cloud_cover_pct', 'mean'),
        weather_condition=('weather_condition', first_mode),
        is_rainy=('is_rainy', first_mode),
        is_snowy=('is_snowy', first_mode),
        is_precipitation=('is_precipitation', first_mode),
        weather_spatial_level=('weather_spatial_level', first_mode),
        is_holiday=('is_holiday', first_mode),
        holiday_name=('holiday_name', first_mode),
        holiday_type=('holiday_type', first_mode),
        is_official_holiday=('is_official_holiday', first_mode),
        is_commercial_event=('is_commercial_event', first_mode),
        is_pre_holiday=('is_pre_holiday', first_mode),
        is_post_holiday=('is_post_holiday', first_mode),
        days_to_nearest_holiday=('days_to_nearest_holiday', 'min'),
        nearest_holiday_name=('nearest_holiday_name', first_mode),
    )
    .reset_index()
)

round_cols = ['revenue', 'avg_transaction_value', 'avg_unit_price', 'temperature_c', 'apparent_temperature_c', 'precipitation_mm']
for col in round_cols:
    df2_hourly[col] = df2_hourly[col].round(3)

print('Dataset 2 hourly aggregate shape:', df2_hourly.shape)
display(df2_hourly.head())
display(df2_hourly['revenue'].describe())

## 6. Feature list cho model và leakage control

In [ ]:
target_col = 'revenue'

time_features = ['month', 'weekday_num', 'hour', 'is_weekend']
product_features = ['product_category']
weather_features = [
    'temperature_c',
    'apparent_temperature_c',
    'relative_humidity',
    'precipitation_mm',
    'wind_speed_10m_kmh',
    'cloud_cover_pct',
    'weather_condition',
    'is_rainy',
    'is_snowy',
    'is_precipitation',
]
holiday_features = [
    'is_holiday',
    'holiday_type',
    'is_official_holiday',
    'is_commercial_event',
    'is_pre_holiday',
    'is_post_holiday',
    'days_to_nearest_holiday',
]
location_features = ['area', 'location_level']

feature_set_no_location = time_features + product_features + weather_features + holiday_features
feature_set_with_location = feature_set_no_location + location_features

leakage_or_post_sale_cols = [
    'quantity',
    'transaction_count',
    'avg_transaction_value',
    'avg_unit_price',
]

print('Target:', target_col)
print('\nFeature set 1 - không dùng location:')
print(feature_set_no_location)
print('\nFeature set 2 - có dùng location:')
print(feature_set_with_location)
print('\nKhông dùng trong model chính vì leakage/post-sale:')
print(leakage_or_post_sale_cols)

## 7. Đánh giá mất cân bằng dữ liệu khi gộp với Dataset 1

In [ ]:
balance_rows = [
    {
        'source_dataset': 'dataset2',
        'transaction_rows': len(df2_clean),
        'hourly_rows': len(df2_hourly),
        'date_min': df2_hourly['date'].min(),
        'date_max': df2_hourly['date'].max(),
    }
]

if DATASET1_FULL_PATH.exists():
    df1_ref = pd.read_csv(DATASET1_FULL_PATH, low_memory=False)
    df1_ref['transaction_datetime'] = pd.to_datetime(df1_ref['timestamp_datetime'] if 'timestamp_datetime' in df1_ref.columns else df1_ref['timestamp'], errors='coerce')
    df1_ref['date'] = df1_ref['transaction_datetime'].dt.normalize()
    df1_ref['hour'] = df1_ref['transaction_datetime'].dt.hour
    df1_ref['area'] = 'city_level_unknown'
    df1_ref['product_category'] = df1_ref['product_category'].astype(str).str.strip()
    df1_ref_hourly_rows = len(df1_ref[['date', 'hour', 'area', 'product_category']].drop_duplicates())
    balance_rows.append(
        {
            'source_dataset': 'dataset1',
            'transaction_rows': len(df1_ref),
            'hourly_rows': df1_ref_hourly_rows,
            'date_min': df1_ref['date'].min(),
            'date_max': df1_ref['date'].max(),
        }
    )

balance_summary = pd.DataFrame(balance_rows)
display(balance_summary)

if set(balance_summary['source_dataset']) == {'dataset1', 'dataset2'}:
    d1_hourly = balance_summary.loc[balance_summary['source_dataset'].eq('dataset1'), 'hourly_rows'].iloc[0]
    d2_hourly = balance_summary.loc[balance_summary['source_dataset'].eq('dataset2'), 'hourly_rows'].iloc[0]
    print(f'Dataset 2 hourly rows / Dataset 1 hourly rows: {d2_hourly / d1_hourly:.2f}x')

print('\nHướng xử lý imbalance sau khi gộp:')
print('1. Baseline: train chung bình thường, nhưng report metric riêng cho Dataset 1 và Dataset 2.')
print('2. Weighted model: dùng sample_weight để Dataset 1 không bị lấn át.')
print('3. Downsample Dataset 2 như experiment phụ để đo sampling variation.')
print('4. Common period experiment: chỉ dùng 2023-01 đến 2023-06 để giảm lệch temporal scope.')
print('5. Location experiment: Model 1 không dùng area/location, Model 2 có dùng area/location.')

## 8. Không lưu CSV ở bước này

Notebook này chỉ tạo các dataframe trong memory:

- `df2_clean`: transaction-level clean schema.
- `df2_hourly`: aggregate-level clean schema.

Chưa xuất file CSV clean để còn kiểm thử và đọc kết quả trước.